# COMP5310 — Stage 2: Data Cleaning Notebook

**Group:** Lab_05_Group_10  
**Dataset:** Hotel Bookings (Dataset C)  
**Purpose:** Produce the final cleaned CSV used by the modelling notebook.

## Directions to run
1. Install Python 3.10 or later.
2. Install dependencies: `pandas`, `numpy`, `matplotlib`, `seaborn`.
3. Place the raw dataset (`hotel_bookings.csv`) in the same folder as this notebook.
4. Run all cells top-to-bottom — the final cell saves `hotel_cleaned_stage2.csv` to disk.
5. The modelling notebook will load this CSV directly.

## What's new vs Stage 1
Stage 1 cleaning is preserved (whitespace, missing values, dropping `company`, outlier removal). Stage 2 adds:
- **Stricter categorical standardisation** (case + typo fixes for `market_segment`, `meal`, `customer_type`, `distribution_channel`)
- **Removal of leakage columns** (`reservation_status`, `reservation_status_date`) — these directly reveal the target
- **Feature engineering** — five new variables derived from existing fields
- **CSV export** for downstream modelling

## Step 1: Import libraries and load data

In [31]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)

In [32]:
df = pd.read_csv('hotel_bookings.csv')
print(f"Initial shape: {df.shape}")
df.head(3)

Initial shape: (119987, 32)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,City Hotel,1,29,2016,February,6,5,6,14,1,0.0,0,BB,BRA,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,9.0,NaN,0,Transient,70.91,0,0,Canceled,2016-01-15
1,Resort Hotel,0,312,2017,March,10,5,2,5,2,0.0,0,HB,DEU,Groups,TA/TO,0,0,0,A,A,0,No Deposit,298.0,NaN,0,Transient-Party,56.00,0,0,Check-Out,2017-03-12
2,Resort Hotel,0,19,2016,February,9,27,0,1,2,0.0,0,BB,PRT,Complementary,TA/TO,0,0,0,A,F,0,No Deposit,5.0,NaN,0,Transient,0.00,0,0,Check-Out,2016-02-28


## Step 2: Inspection — quick recap from Stage 1

In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119987 entries, 0 to 119986
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119987 non-null  object 
 1   is_canceled                     119987 non-null  int64  
 2   lead_time                       119987 non-null  int64  
 3   arrival_date_year               119987 non-null  int64  
 4   arrival_date_month              119987 non-null  object 
 5   arrival_date_week_number        119987 non-null  int64  
 6   arrival_date_day_of_month       119987 non-null  int64  
 7   stays_in_weekend_nights         119987 non-null  int64  
 8   stays_in_week_nights            119987 non-null  int64  
 9   adults                          119987 non-null  int64  
 10  children                        119805 non-null  float64
 11  babies                          119987 non-null  int64  
 12  meal            

In [34]:
# Missing values summary
missing_counts = df.isnull().sum()
missing_percent = (missing_counts / len(df)) * 100
missing_summary = pd.DataFrame({
    'Missing Count': missing_counts,
    'Missing %': missing_percent.round(2)
}).sort_values(by='Missing Count', ascending=False)
print(missing_summary[missing_summary['Missing Count'] > 0])

                      Missing Count  Missing %
company                      113162      94.31
agent                         16541      13.79
country                         628       0.52
children                        182       0.15
distribution_channel            164       0.14
customer_type                   154       0.13
meal                            138       0.12
market_segment                  129       0.11


In [35]:
# Inspect categorical inconsistencies
for col in ['hotel', 'market_segment', 'distribution_channel', 'meal', 'customer_type', 'deposit_type']:
    print(f"\n{col} unique values: {df[col].unique()}")


hotel unique values: ['City Hotel' 'Resort Hotel' 'ResortHotel' 'CityHotel']

market_segment unique values: ['Online TA' 'Groups' 'Complementary' 'Direct' nan 'Offline TA/TO'
 'Corporate' 'Aviation' 'Corporate ' 'groups' 'Direc' 'offline ta/to'
 'COMPLEMENTARY' 'Undefined ' 'Undefined']

distribution_channel unique values: ['TA/TO' 'Direct' 'Corporate' 'TA/TO ' 'GDS' 'Direct ' nan 'Corporate '
 'Undefined' 'gds']

meal unique values: ['BB' 'HB' 'SC' 'FB' 'Undefined' nan 'HB ' 'bb' 'Undefined ' 'FB ' 'SC ']

customer_type unique values: ['Transient' 'Transient-Party' 'Contract' 'Group' nan 'Transient '
 'Transient-Party ' 'contract' 'GROUP']

deposit_type unique values: ['No Deposit' 'Non Refund' 'Refundable']


## Step 3: Data Cleaning

### 3.1 Strip whitespace and standardise case

In [36]:
# Strip whitespace from all string columns.
# IMPORTANT: we use .str.strip() WITHOUT .astype(str) to avoid converting
# real NaN values into the string 'nan' (a known pandas pitfall on some
# versions). .str methods on pandas Series natively skip NaN.
text_cols = ['hotel', 'meal', 'country', 'market_segment', 'distribution_channel',
             'reserved_room_type', 'assigned_room_type', 'deposit_type',
             'customer_type', 'reservation_status']
for c in text_cols:
    df[c] = df[c].str.strip()

# Standardise case (these .str methods also skip NaN automatically)
df['market_segment']       = df['market_segment'].str.title()
df['customer_type']        = df['customer_type'].str.title()
df['distribution_channel'] = df['distribution_channel'].str.title()
df['meal']                 = df['meal'].str.upper()

# Defensive: if any 'nan'/'NaN'/'NAN' strings have leaked in, restore them as real NaN
for c in text_cols:
    df[c] = df[c].replace({'Nan': np.nan, 'NaN': np.nan, 'nan': np.nan, 'NAN': np.nan})

print("After case standardisation:")
print("  market_segment:", sorted(df['market_segment'].dropna().unique().tolist()))
print("  customer_type:", sorted(df['customer_type'].dropna().unique().tolist()))
print("  meal:", sorted(df['meal'].dropna().unique().tolist()))
print("  Real NaN counts after this step:")
for c in ['meal', 'market_segment', 'customer_type', 'distribution_channel']:
    print(f"    {c}: {df[c].isna().sum()}")

After case standardisation:
  market_segment: ['Aviation', 'Complementary', 'Corporate', 'Direc', 'Direct', 'Groups', 'Offline Ta/To', 'Online Ta', 'Undefined']
  customer_type: ['Contract', 'Group', 'Transient', 'Transient-Party']
  meal: ['BB', 'FB', 'HB', 'SC', 'UNDEFINED']
  Real NaN counts after this step:
    meal: 138
    market_segment: 129
    customer_type: 154
    distribution_channel: 164


### 3.2 Fix typos and undefined categories
After title-casing, several values still need targeted fixes:
- `'Direc'` is clearly a truncated `'Direct'`
- `'Online Ta'` and `'Offline Ta/To'` should keep TA/TO uppercase as in the original taxonomy
- `'Undefined'` in `market_segment` is mapped to the mode (`Online TA`) — only a handful of rows
- `'Undefined'` in `meal` is mapped to `'SC'` (Self-Catering) per the dataset description

In [37]:
segment_fix = {
    'Online Ta': 'Online TA',
    'Offline Ta/To': 'Offline TA/TO',
    'Direc': 'Direct',
    'Undefined': 'Online TA'
}
df['market_segment'] = df['market_segment'].replace(segment_fix)

df['distribution_channel'] = df['distribution_channel'].replace({
    'Ta/To': 'TA/TO', 'Gds': 'GDS', 'Undefined': 'TA/TO'
})

df['meal'] = df['meal'].replace({'UNDEFINED': 'SC'})

# Fix hotel column: 'CityHotel' / 'ResortHotel' (no space) appear as separate values
df['hotel'] = df['hotel'].replace({'CityHotel': 'City Hotel', 'ResortHotel': 'Resort Hotel'})

print("hotel unique:", sorted(df['hotel'].dropna().unique().tolist()))
print("market_segment unique:", sorted(df['market_segment'].dropna().unique().tolist()))
print("distribution_channel unique:", sorted(df['distribution_channel'].dropna().unique().tolist()))
print("meal unique:", sorted(df['meal'].dropna().unique().tolist()))
print("customer_type unique:", sorted(df['customer_type'].dropna().unique().tolist()))

hotel unique: ['City Hotel', 'Resort Hotel']
market_segment unique: ['Aviation', 'Complementary', 'Corporate', 'Direct', 'Groups', 'Offline TA/TO', 'Online TA']
distribution_channel unique: ['Corporate', 'Direct', 'GDS', 'TA/TO']
meal unique: ['BB', 'FB', 'HB', 'SC']
customer_type unique: ['Contract', 'Group', 'Transient', 'Transient-Party']


### 3.3 Handle missing values

In [38]:
fill_values = {
    'children': 0,
    'country': 'Unknown',
    'agent': 0,
    'market_segment': df['market_segment'].mode()[0],
    'customer_type': df['customer_type'].mode()[0],
    'meal': df['meal'].mode()[0],
    'distribution_channel': df['distribution_channel'].mode()[0]
}
df.fillna(value=fill_values, inplace=True)
print("Remaining missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Remaining missing values:
company    113162
dtype: int64


### 3.4 Drop columns with high missing rate

In [39]:
# 'company' has ~94% missing, dropped
df.drop(columns=['company'], inplace=True, errors='ignore')
print(f"Shape after dropping 'company': {df.shape}")

Shape after dropping 'company': (119987, 31)


### 3.5 Convert data types

In [40]:
df['reservation_status_date'] = pd.to_datetime(
    df['reservation_status_date'], format='mixed', errors='coerce'
)

### 3.6 Remove invalid rows and outliers

In [41]:
# Remove bookings with zero total guests (logically impossible)
no_guests_mask = (df['adults'] == 0) & (df['children'] == 0) & (df['babies'] == 0)
print(f"Removing {no_guests_mask.sum()} zero-guest rows")
df = df[~no_guests_mask].copy()

# Remove extreme outliers in adults (>10 — fewer than 0.01% of data, all cancelled)
df = df[df['adults'] <= 10]

# Remove invalid ADR values
df = df[(df['adr'] >= 0) & (df['adr'] < 5000)]

print(f"Shape after outlier removal: {df.shape}")

Removing 180 zero-guest rows
Shape after outlier removal: (119793, 31)


## Step 4: Stage 2 additions

### 4.1 Drop leakage columns ⚠️
`reservation_status` directly states whether a booking was cancelled (`Canceled`/`Check-Out`/`No-Show`), and `reservation_status_date` records when that status was set. Both are recorded **after** the booking outcome is known, so using them as features would constitute target leakage and inflate model performance artificially. We drop them before any modelling.

In [42]:
leakage_cols = ['reservation_status', 'reservation_status_date']
df = df.drop(columns=leakage_cols)
print(f"Dropped leakage columns: {leakage_cols}")
print(f"Shape: {df.shape}")

Dropped leakage columns: ['reservation_status', 'reservation_status_date']
Shape: (119793, 29)


### 4.2 Feature engineering
We derive five new features from existing columns. All are computable at the time of reservation, so none introduce leakage:

| Feature | Definition | Rationale |
|---|---|---|
| `total_nights` | `stays_in_weekend_nights + stays_in_week_nights` | Captures total stay length in one variable |
| `total_guests` | `adults + children + babies` | Captures party size |
| `has_special_request` | `1 if total_of_special_requests > 0 else 0` | Binary version of an already-strong predictor; useful for tree-based models |
| `is_summer_arrival` | `1 if arrival month in {Jun, Jul, Aug} else 0` | Stage 1 EDA showed June peaks at 41.5% cancellation rate |
| `prev_cancel_rate` | `previous_cancellations / (previous_cancellations + previous_bookings_not_canceled + 1)` | Smoothed historical cancel rate per guest; +1 in denominator avoids divide-by-zero |

In [43]:
df['total_nights']        = df['stays_in_weekend_nights'] + df['stays_in_week_nights']
df['total_guests']        = df['adults'] + df['children'] + df['babies']
df['has_special_request'] = (df['total_of_special_requests'] > 0).astype(int)

summer_months = ['June', 'July', 'August']
df['is_summer_arrival'] = df['arrival_date_month'].astype(str).isin(summer_months).astype(int)

df['prev_cancel_rate'] = (
    df['previous_cancellations'] /
    (df['previous_cancellations'] + df['previous_bookings_not_canceled'] + 1)
)

print("New features added.")
df[['total_nights', 'total_guests', 'has_special_request',
    'is_summer_arrival', 'prev_cancel_rate']].head()

New features added.


,total_nights,total_guests,has_special_request,is_summer_arrival,prev_cancel_rate
0,20,1.0,0,0,0.0
1,7,2.0,0,0,0.0
2,1,2.0,0,0,0.0
3,2,2.0,0,1,0.0
4,5,2.0,1,1,0.0


### 4.3 Final sanity checks

In [44]:
print("Final shape:", df.shape)
print("\nTarget distribution (is_canceled):")
print(df['is_canceled'].value_counts(normalize=True).round(4))
print("\nRemaining missing values (should be 0):")
print(df.isnull().sum().sum())
print("\nColumn list:")
print(df.columns.tolist())

Final shape: (119793, 34)

Target distribution (is_canceled):
is_canceled
0    0.6293
1    0.3707
Name: proportion, dtype: float64

Remaining missing values (should be 0):
0

Column list:
['hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'days_in_waiting_list', 'customer_type', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'total_nights', 'total_guests', 'has_special_request', 'is_summer_arrival', 'prev_cancel_rate']


## Step 5: Save cleaned dataset

In [45]:
OUTPUT_PATH = 'hotel_cleaned_stage2.csv'
df.to_csv(OUTPUT_PATH, index=False)
print(f"Cleaned dataset saved to: {OUTPUT_PATH}")
print(f"Final shape: {df.shape}")
print(f"Final columns ({len(df.columns)}): {df.columns.tolist()}")

Cleaned dataset saved to: hotel_cleaned_stage2.csv
Final shape: (119793, 34)
Final columns (34): ['hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'days_in_waiting_list', 'customer_type', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'total_nights', 'total_guests', 'has_special_request', 'is_summer_arrival', 'prev_cancel_rate']


## Summary of changes vs raw data

| Step | Action | Rows | Cols |
|---|---|---|---|
| Raw | Loaded `hotel_bookings.csv` | 119,987 | 32 |
| 3.1–3.2 | Cleaned categorical columns (whitespace, case, typos) | 119,987 | 32 |
| 3.3 | Filled missing values (mode/0/'Unknown') | 119,987 | 32 |
| 3.4 | Dropped `company` (94% missing) | 119,987 | 31 |
| 3.5 | Converted `reservation_status_date` to datetime | 119,987 | 31 |
| 3.6 | Removed zero-guest rows + extreme outliers | 119,793 | 31 |
| 4.1 | Dropped 2 leakage columns | 119,793 | 29 |
| 4.2 | Added 5 engineered features | 119,793 | 34 |

**Final dataset:** 119,793 rows × 34 columns, target `is_canceled` (37% positive rate), no missing values, no leakage.

The next step is the modelling notebook — load `hotel_cleaned_stage2.csv` and fit Logistic Regression, Random Forest, and K-Means.